In [0]:
import time
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql import types as T

dbutils.widgets.text("catalogo", "workspace")

catalogo = dbutils.widgets.get("catalogo")

spark.sql(f"USE CATALOG {catalogo}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

TABELA_ORIGEM  = "bronze.tb_tickets"
TABELA_DESTINO = "silver.tb_tickets"

print(f"Catálogo em uso : {catalogo}")
print(f"Origem          : {TABELA_ORIGEM}")
print(f"Destino         : {TABELA_DESTINO}")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    regexp_replace,
    to_date,
    current_date,
    floor,
    datediff,
    when,
    length
)

In [0]:
# Todos os campos chegam como STRING da Bronze (inferSchema=false).
# A tipagem correta é feita neste notebook de forma controlada e rastreável.
inicio_total = time.time()
execucao_id  = datetime.now().strftime("%Y%m%d_%H%M%S")

df_bronze = spark.table(TABELA_ORIGEM)

total_bronze = df_bronze.count()
print(f"Registros lidos da Bronze: {total_bronze:,}")
print(f"Colunas: {df_bronze.columns}")

In [0]:
print((df_bronze.count(), len(df_bronze.columns)))

In [0]:
display(df_bronze)

In [0]:
df_silver = df_bronze

In [0]:
# Conta quantos IDs estão repetidos
total = df_silver.groupBy("ticket_id").count().filter("count > 1").count()

print(f"Total de ticket_id duplicados: {total}")

In [0]:
print("Contagem de valores NULOS por coluna:")

expressoes_nulos = [
    F.sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df_silver.columns
]

df_silver.select(*expressoes_nulos).display()

In [0]:
# Traz apenas os valores únicos da coluna origem
df_silver.select("tipo_problema").distinct().display()

In [0]:
df_silver = df_silver\
    .withColumn("data_abertura", F.col("data_abertura").cast(TimestampType())) \
    .withColumn("data_resolucao", F.col("data_resolucao").cast(TimestampType())) \
    .withColumn("tempo_resolucao_horas", F.col("tempo_resolucao_horas").cast(DoubleType())) \
    .withColumn("nota_avaliacao", F.col("nota_avaliacao").cast("int")) \
    .withColumn(
        "tipo_problema_padronizado",
        F.when(F.lower(F.col("tipo_problema")).rlike("prod|p3od|p3oduto"), "Produto")
         .when(F.lower(F.col("tipo_problema")).rlike("pag|pay|p4ga|p4gamento"), "Pagamento")
         .when(F.lower(F.col("tipo_problema")).rlike("entr|del|3ntr|3ntrega"), "Entrega")
         .when(F.lower(F.col("tipo_problema")).rlike("reem|ref|r3em|reemb"), "Reembolso")
         .otherwise("Outros")
    ) \
    .withColumn(
        "tempo_resolucao_dias",
        F.round(F.col("tempo_resolucao_horas") / 24, 2)
    )

display(df_silver)

In [0]:
df_silver.select(
    F.max("data_abertura")
).show()

In [0]:
# Verificado o máx e minimo das avaliações
df_silver.select(
    F.max("nota_avaliacao")
).show()

df_silver.select(
    F.min("nota_avaliacao")
).show()

In [0]:
df_datas_invalidas = df_silver.filter(
    col("data_abertura") > col("data_resolucao")
)

print("Quantidade de registros inválidos:")
print(df_datas_invalidas.count())

df_datas_invalidas.display()

In [0]:
df_silver = df_silver.withColumn(
    "tipo_problema",
    F.col("tipo_problema_padronizado")
)

In [0]:
# Traz apenas os valores únicos da coluna origem
df_silver.select("tipo_problema_padronizado").distinct().display()

In [0]:
df_silver = df_bronze.withColumn("data_abertura", F.col("data_abertura").cast(TimestampType())) \
                     .withColumn("data_resolucao", F.col("data_resolucao").cast(TimestampType())) \
                     .withColumn("tempo_resolucao_horas", F.col("tempo_resolucao_horas").cast(DoubleType())) \
                     .withColumn("nota_avaliacao", F.col("nota_avaliacao").cast("int"))

# Padronização do tipo_problema (usando o que vimos antes)
df_silver = df_silver.withColumn(
    "tipo_problema_padronizado",
    F.when(F.lower(F.col("tipo_problema")).rlike("prod|p3od"), "Produto")
    .when(F.lower(F.col("tipo_problema")).rlike("pag|pay|p4ga"), "Pagamento")
    .when(F.lower(F.col("tipo_problema")).rlike("entr|del|3ntr"), "Entrega")
    .when(F.lower(F.col("tipo_problema")).rlike("reem|ref|r3em"), "Reembolso")
    .otherwise("Outros")
)

# Criando a coluna de dias
df_silver = df_silver.withColumn(
    "tempo_resolucao_dias", 
    F.round(F.col("tempo_resolucao_horas") / 24, 2)
)

display(df_silver)

In [0]:
df_silver = df_silver.drop("tipo_problema")

In [0]:
# (Data é nula MAS o tempo de horas não é) OU (Data NÃO é nula MAS o tempo é nulo)
inconsistencias = df_silver.filter(
    (F.col("data_resolucao").isNull() & F.col("tempo_resolucao_horas").isNotNull()) |
    (F.col("data_resolucao").isNotNull() & F.col("tempo_resolucao_horas").isNull())
)

# Conta quantas linhas deram erro
total_erros = inconsistencias.count()

if total_erros == 0:
    print(" Sucesso: Todas as colunas nulas estão consistentes!")
else:
    print(f" Atenção: Foram encontradas {total_erros} linhas inconsistentes.")
    display(inconsistencias)

In [0]:
#Salvando a tabela
df_silver_tickets = df_silver
df_silver_tickets.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(TABELA_DESTINO)

print(f"Camada Silver finalizada com o DataFrame: df_silver_tickets")
print(f"Tabela salva em: {TABELA_DESTINO}")

In [0]:
df_silver.select("agente_suporte") \
    .distinct() \
    .orderBy("agente_suporte") \
    .display()

In [0]:
display(df_silver_tickets.limit(16))